# Maia Lite — Corpus v8: Full Validation & Tokenization Audit

**Colab | Read-only source corpus | Resumable | 2026-10-24**

This notebook independently validates **all committed gzip chunks** from `acquisition_v8_5_001`, performs **global exact deduplication**, optionally counts **real tokens using the frozen Maia tokenizer**, and performs a separate language-ID sample.

### Safety and interpretation
- Source chunks remain untouched in `quarantine_pending_rights_review/`.
- No Hugging Face dataset download or acquisition is started.
- An interrupted run resumes from a **checkpointed SQLite audit database**, never from an in-memory-only `set`.
- **No tokenizer substitution**: configure the exact Maia tokenizer locally; otherwise real-token verification remains **NOT RUN**.
- Audit completion does **not** establish rights to train, redistribution permission, near-duplicate removal, benchmark decontamination, or suitability for training.
- Snapshotting a multi-GB SQLite database to Drive can take time; progress between snapshots can be replayed after a runtime reset.

**Run in order.** The full scan may require hours. Use a Colab runtime with enough local SSD space. The first cells only configure and inventory; the full scan runs only in its dedicated cell.

In [1]:
%pip -q install pandas matplotlib psutil transformers tokenizers langid

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import os, sys, time, json, gzip, re, hashlib, sqlite3, shutil, random, gc
import pandas as pd
from IPython.display import display
import psutil
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive', timeout_ms=300000)
    else: print('Google Drive already mounted')
except ImportError:
    print('Non-Colab environment: adjust ROOT below')

ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
RUN_ID='acquisition_v8_5_001'
RUN_DIR=ROOT/'corpus_v8'/'runs'/RUN_ID
SOURCE=RUN_DIR/'quarantine_pending_rights_review'
OUT = RUN_DIR / 'quality_audit_full_with_tokenizer_v1'
OUT.mkdir(parents=True, exist_ok=True)

# Explicit switches
ENABLE_FULL_SCAN=True          # Change to True before the full-scan cell
ENABLE_REAL_TOKEN_COUNT=True   # Requires a local, exact Maia tokenizer path
TOKENIZER_PATH='/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer'  # e.g. /content/drive/MyDrive/.../tokenizador
ENABLE_LANGUAGE_SAMPLE=True

# Persistence: checkpoint database snapshots are atomic and resumable
CHECKPOINT_EVERY_CHUNKS=200
CHECKPOINT_EVERY_MINUTES=30
PRINT_EVERY_CHUNKS=25
MAX_ERRORS_LOGGED=500
LANGUAGE_SAMPLE_FILES_PER_JOB=24
LANGUAGE_SAMPLE_DOCS_PER_FILE=20

assert SOURCE.is_dir(),f'Corpus source not found: {SOURCE}'
assert CHECKPOINT_EVERY_CHUNKS>=1
print('SOURCE (read only):',SOURCE)
print('REPORTS:',OUT)
print('RAM available (GB):',round(psutil.virtual_memory().available/2**30,2))
print('Local disk free (GB):',round(shutil.disk_usage('/content' if Path('/content').exists() else '/tmp').free/2**30,2))

Mounted at /content/drive
SOURCE (read only): /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001/quarantine_pending_rights_review
REPORTS: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001/quality_audit_full_with_tokenizer_v1
RAM available (GB): 10.54
Local disk free (GB): 205.31


## 1. Frozen inventory and safety checks

Inventory is built from filenames only. Compare with the previously observed **21,021 chunks**, **8,180,848,581 proxy tokens**, and **10,101,231 documents**; these are *reference figures*, not assumed audit results. Abort if acquisition is still modifying the source directory.

In [5]:
JOBS=[('fineweb2_pt','pt','web',2624000000),('fineweb2_es','es','web',1640000000),('fineweb_edu_en','en','web',2296000000),('wikipedia_pt','pt','wikipedia',656000000),('wikipedia_en','en','wikipedia',574000000),('wikipedia_es','es','wikipedia',410000000)]
JOB_META={j:dict(language=lang,family=fam,target=target) for j,lang,fam,target in JOBS}
PATTERN=re.compile(r'^(.*?)_chunk_(\d+)\.jsonl\.gz$')
files_by_job={j:sorted(SOURCE.glob(f'{j}_chunk_*.jsonl.gz')) for j in JOB_META}
files=[(j,p) for j in JOB_META for p in files_by_job[j]]
unknown=[p.name for p in SOURCE.glob('*.jsonl.gz') if not any(p.name.startswith(j+'_chunk_') for j in JOB_META)]
inv=[]
for j,ps in files_by_job.items():
    ids=[int(PATTERN.match(p.name).group(2)) for p in ps if PATTERN.match(p.name)]
    inv.append(dict(job_id=j,language=JOB_META[j]['language'],chunks=len(ps),duplicate_ids=len(ids)-len(set(ids)),compressed_gb=round(sum(p.stat().st_size for p in ps)/1e9,3)))
display(pd.DataFrame(inv))
print('Chunks:',len(files),'Unknown files:',unknown[:10])
assert len(files)>0 and not unknown,'Unexpected source inventory; review manually'
assert all(row['duplicate_ids']==0 for row in inv),'Duplicate chunk indices'
# Manifest fingerprint: paths and file sizes/mtime, to detect source changes across restarts.
manifest_rows=[dict(job_id=j,name=p.name,size=p.stat().st_size,mtime_ns=p.stat().st_mtime_ns) for j,p in files]
inventory_digest=hashlib.sha256(json.dumps(manifest_rows,sort_keys=True).encode()).hexdigest()
print('Inventory SHA256:',inventory_digest)
print('Stop acquisition before continuing; do not edit source files.')

,job_id,language,chunks,duplicate_ids,compressed_gb
0,fineweb2_pt,pt,7510,0,4.927
1,fineweb2_es,es,4442,0,2.990
2,fineweb_edu_en,en,4151,0,3.884
3,wikipedia_pt,pt,1944,0,1.100
4,wikipedia_en,en,1662,0,0.976
5,wikipedia_es,es,1312,0,0.704


Chunks: 21021 Unknown files: []
Inventory SHA256: bd8d1a6ab4a5b59d4967ad2976bccfcd66c3d9980b9b864699a4fe7f61cd4403
Stop acquisition before continuing; do not edit source files.


## 2. Persistent audit database

SQLite is stored on the **Colab local SSD** for performance, with atomic snapshots on Drive. Each gzip chunk is processed in a single transaction. On a crash, the last snapshot is restored and any work since that snapshot is replayed. The `seen` table deduplicates SHA-256 hashes **across all six sources**.

**Note:** Database snapshots may be large. The exact-tokenizer configuration and source inventory are locked into the checkpoint. Changing either requires a **new audit directory** (do not silently mix incompatible results).

In [7]:
import json
from pathlib import Path

print("OUT:", OUT)
print("CONFIG:", CONFIG)
print("SNAP:", SNAP)

prior = json.loads(CONFIG.read_text(encoding="utf-8"))

print("\nPREVIOUS CONFIGURATION:")
print(json.dumps(prior, indent=2, ensure_ascii=False))

print("\nCURRENT CONFIGURATION:")
print(json.dumps(CURRENT_CONFIG, indent=2, ensure_ascii=False))

print("\nDIFFERENCES:")
for key in sorted(set(prior) | set(CURRENT_CONFIG)):
    if prior.get(key) != CURRENT_CONFIG.get(key):
        print(
            f"{key}: "
            f"previous={prior.get(key)!r}, "
            f"current={CURRENT_CONFIG.get(key)!r}"
        )

print("\nCHECKPOINT EXISTS:", SNAP.exists())

if SNAP.exists():
    print("CHECKPOINT SIZE:", SNAP.stat().st_size, "bytes")

OUT: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001/quality_audit_full_with_tokenizer_v1
CONFIG: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001/quality_audit_full_with_tokenizer_v1/audit_config.json
SNAP: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001/quality_audit_full_with_tokenizer_v1/audit_state.sqlite3

PREVIOUS CONFIGURATION:
{
  "run_id": "acquisition_v8_5_001",
  "inventory_sha256": "bd8d1a6ab4a5b59d4967ad2976bccfcd66c3d9980b9b864699a4fe7f61cd4403",
  "real_token_count": true,
  "tokenizer_path": ""
}

CURRENT CONFIGURATION:
{
  "run_id": "acquisition_v8_5_001",
  "inventory_sha256": "bd8d1a6ab4a5b59d4967ad2976bccfcd66c3d9980b9b864699a4fe7f61cd4403",
  "real_token_count": true,
  "tokenizer_path": "/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer"
}

DIFFERENCES:
tokenizer_path: previous='', current='/content/drive/MyDrive/Colab_LLMs/Maia_L

In [8]:
import sqlite3

connection = sqlite3.connect(
    f"file:{SNAP}?mode=ro",
    uri=True
)

tables = connection.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""").fetchall()

print("TABLES:")
for (tableName,) in tables:
    print(" -", tableName)

print("\nDATABASE INTEGRITY:")
print(connection.execute("PRAGMA quick_check").fetchone()[0])

connection.close()

TABLES:
 - completed
 - error_examples
 - issue_counts
 - seen

DATABASE INTEGRITY:
ok


In [10]:
import sqlite3

connection = sqlite3.connect(
    f"file:{SNAP}?mode=ro",
    uri=True
)

print("=" * 60)
print("MAIA CORPUS V8 — CHECKPOINT RECOVERY")
print("=" * 60)

for tableName in ["completed", "error_examples", "issue_counts", "seen"]:
    count = connection.execute(
        f'SELECT COUNT(*) FROM "{tableName}"'
    ).fetchone()[0]

    print(f"{tableName:20s}: {count:,}")

print("\nTABLE SCHEMAS:")

for tableName in ["completed", "issue_counts"]:
    print(f"\n{tableName}:")
    columns = connection.execute(
        f'PRAGMA table_info("{tableName}")'
    ).fetchall()

    for column in columns:
        print(f"  {column[1]} ({column[2]})")

print("\nCOMPLETED SAMPLE:")

rows = connection.execute(
    'SELECT * FROM "completed" LIMIT 3'
).fetchall()

for row in rows:
    print(row)

connection.close()

MAIA CORPUS V8 — CHECKPOINT RECOVERY
completed           : 21,021
error_examples      : 0
issue_counts        : 0
seen                : 10,101,231

TABLE SCHEMAS:

completed:
  name (TEXT)
  job_id (TEXT)
  compressed_size (INTEGER)
  mtime_ns (INTEGER)
  docs (INTEGER)
  proxy_tokens (INTEGER)
  real_tokens (INTEGER)
  duplicates (INTEGER)
  issues (INTEGER)
  completed_utc (TEXT)

issue_counts:
  kind (TEXT)
  n (INTEGER)

COMPLETED SAMPLE:
('fineweb2_pt_chunk_000000.jsonl.gz', 'fineweb2_pt', 610767, 1792362354000000000, 440, 320244, 299251, 0, 0, '2026-10-24T17:40:28.392135+00:00')
('fineweb2_pt_chunk_000001.jsonl.gz', 'fineweb2_pt', 720127, 1792362533000000000, 500, 376661, 363102, 0, 0, '2026-10-24T17:40:31.135579+00:00')
('fineweb2_pt_chunk_000002.jsonl.gz', 'fineweb2_pt', 757953, 1792362534000000000, 500, 409453, 385818, 0, 0, '2026-10-24T17:40:32.934638+00:00')


In [11]:
import sqlite3

EXPECTED_CHUNKS = 21021
EXPECTED_DOCS = 10101231

connection = sqlite3.connect(
    f"file:{SNAP}?mode=ro",
    uri=True
)

connection.row_factory = sqlite3.Row

print("=" * 76)
print("MAIA CORPUS V8 — FINAL AUDIT RECOVERY REPORT")
print("=" * 76)

summary = connection.execute("""
    SELECT
        COUNT(*) AS chunks,
        SUM(docs) AS documents,
        SUM(proxy_tokens) AS proxy_tokens,
        SUM(real_tokens) AS real_tokens,
        SUM(duplicates) AS duplicates,
        SUM(issues) AS issues,
        SUM(CASE WHEN real_tokens IS NULL THEN 1 ELSE 0 END)
            AS missing_token_counts
    FROM completed
""").fetchone()

print("\nGLOBAL TOTALS")

for key in summary.keys():
    print(f"{key:24s}: {summary[key]:,}")

print("\nPER-DATASET TOTALS")

rows = connection.execute("""
    SELECT
        job_id,
        COUNT(*) AS chunks,
        SUM(docs) AS documents,
        SUM(proxy_tokens) AS proxy_tokens,
        SUM(real_tokens) AS real_tokens,
        SUM(duplicates) AS duplicates,
        SUM(issues) AS issues
    FROM completed
    GROUP BY job_id
    ORDER BY job_id
""").fetchall()

print(
    f"{'Dataset':22s}"
    f"{'Chunks':>10s}"
    f"{'Documents':>14s}"
    f"{'Proxy tokens':>18s}"
    f"{'Real tokens':>18s}"
    f"{'Duplicates':>13s}"
    f"{'Issues':>10s}"
)

for row in rows:
    print(
        f"{row['job_id']:22s}"
        f"{row['chunks']:10,d}"
        f"{row['documents']:14,d}"
        f"{row['proxy_tokens']:18,d}"
        f"{row['real_tokens']:18,d}"
        f"{row['duplicates']:13,d}"
        f"{row['issues']:10,d}"
    )

print("\nTOKENIZATION STATISTICS")

proxy = summary["proxy_tokens"]
real = summary["real_tokens"]

if proxy and real:
    print(f"Real / proxy ratio: {real / proxy:.6f}")
    print(f"Proxy / real ratio: {proxy / real:.6f}")
    print(f"Real tokens:        {real / 1e9:.6f} billion")

print("\nCONSISTENCY CHECKS")

checks = {
    "Expected chunks": summary["chunks"] == EXPECTED_CHUNKS,
    "Expected documents": summary["documents"] == EXPECTED_DOCS,
    "Real token counts present": summary["missing_token_counts"] == 0,
    "No reported duplicates": summary["duplicates"] == 0,
    "No reported issues": summary["issues"] == 0,
}

for label, passed in checks.items():
    print(f"{label:30s}: {'PASS' if passed else 'REVIEW'}")

print("\nDATABASE INTEGRITY")
print(connection.execute("PRAGMA quick_check").fetchone()[0])

connection.close()

MAIA CORPUS V8 — FINAL AUDIT RECOVERY REPORT

GLOBAL TOTALS
chunks                  : 21,021
documents               : 10,101,231
proxy_tokens            : 8,180,848,581
real_tokens             : 7,590,554,035
duplicates              : 0
issues                  : 0
missing_token_counts    : 0

PER-DATASET TOTALS
Dataset                   Chunks     Documents      Proxy tokens       Real tokens   Duplicates    Issues
fineweb2_es                4,442     2,137,958     1,640,000,722     1,498,135,227            0         0
fineweb2_pt                7,510     3,629,088     2,624,000,361     2,443,283,180            0         0
fineweb_edu_en             4,151     1,957,769     2,296,000,629     2,018,075,943            0         0
wikipedia_en               1,662       801,546       574,000,009       568,999,278            0         0
wikipedia_es               1,312       636,939       410,001,215       410,870,248            0         0
wikipedia_pt               1,944       937,931    

In [9]:
LOCAL = Path('/content/maia_full_audit_with_tokenizer.sqlite3')
SNAP = OUT / 'audit_state.sqlite3'
CONFIG=OUT/'audit_config.json'
CURRENT_CONFIG=dict(run_id=RUN_ID,inventory_sha256=inventory_digest,real_token_count=ENABLE_REAL_TOKEN_COUNT,tokenizer_path=str(TOKENIZER_PATH) if ENABLE_REAL_TOKEN_COUNT else None)
if CONFIG.exists():
    prior=json.loads(CONFIG.read_text(encoding='utf-8'))
    assert prior==CURRENT_CONFIG, f'Checkpoint configuration differs. Existing: {prior}. Use a separate OUT directory for a new mode.'
else:
    CONFIG.write_text(json.dumps(CURRENT_CONFIG,indent=2),encoding='utf-8')

if not LOCAL.exists() and SNAP.exists():
    print('Restoring database snapshot from Drive...')
    shutil.copy2(SNAP,LOCAL)
if LOCAL.exists() and not SNAP.exists():
    print('Local DB exists without Drive snapshot; use cautiously if this is a new session.')
conn=sqlite3.connect(LOCAL,timeout=120)
conn.execute('PRAGMA journal_mode=DELETE')
conn.execute('PRAGMA synchronous=FULL')
conn.execute('PRAGMA temp_store=FILE')
conn.execute('CREATE TABLE IF NOT EXISTS seen(digest TEXT PRIMARY KEY) WITHOUT ROWID')
conn.execute("""CREATE TABLE IF NOT EXISTS completed (
    name TEXT PRIMARY KEY, job_id TEXT NOT NULL, compressed_size INTEGER NOT NULL,
    mtime_ns INTEGER NOT NULL, docs INTEGER NOT NULL, proxy_tokens INTEGER NOT NULL,
    real_tokens INTEGER, duplicates INTEGER NOT NULL, issues INTEGER NOT NULL,
    completed_utc TEXT NOT NULL)""")
conn.execute('CREATE TABLE IF NOT EXISTS issue_counts (kind TEXT PRIMARY KEY, n INTEGER NOT NULL)')
conn.execute('CREATE TABLE IF NOT EXISTS error_examples (id INTEGER PRIMARY KEY, file TEXT, line INTEGER, kind TEXT, detail TEXT)')
conn.commit()
completed=conn.execute('SELECT COUNT(*) FROM completed').fetchone()[0]
print('Completed chunks in local DB:',completed,'/',len(files))
print('DB size (GB):',round(LOCAL.stat().st_size/1e9,3))

def save_checkpoint():
    # SQLite backup creates a consistent, standalone database; then atomic Drive rename.
    conn.commit()
    tmp=OUT/'audit_state.sqlite3.pending'
    if tmp.exists():tmp.unlink()
    dest=sqlite3.connect(tmp)
    try:
        conn.backup(dest,pages=1000,sleep=0.1)
    finally: dest.close()
    os.replace(tmp,SNAP)
    (OUT/'checkpoint_status.json').write_text(json.dumps({
        'saved_utc':datetime.now(timezone.utc).isoformat(),
        'completed_chunks':conn.execute('SELECT COUNT(*) FROM completed').fetchone()[0],
        'db_bytes':SNAP.stat().st_size,'inventory_sha256':inventory_digest},indent=2),encoding='utf-8')
    print('Checkpoint saved:',SNAP,'chunks:',conn.execute('SELECT COUNT(*) FROM completed').fetchone()[0],flush=True)

AssertionError: Checkpoint configuration differs. Existing: {'run_id': 'acquisition_v8_5_001', 'inventory_sha256': 'bd8d1a6ab4a5b59d4967ad2976bccfcd66c3d9980b9b864699a4fe7f61cd4403', 'real_token_count': True, 'tokenizer_path': ''}. Use a separate OUT directory for a new mode.

## 3. Optional frozen tokenizer (exact, local)

The pretraining tokenizer must be loaded **without network access or fallback**. For a reproducible full count, specify the actual Maia Lite tokenizer directory and set `ENABLE_REAL_TOKEN_COUNT=True` **before running the database cell**. If already auditing without tokenization, finish that audit; start a new output directory for a tokenizer-enabled full scan.

Tokenizer counts use `add_special_tokens=False`, count the **entire text**, and exclude any packing/separator overhead. These are raw text-token counts, not the final number of packed training tokens.

In [ ]:
from pathlib import Path

base = Path('/content/drive/MyDrive/Colab_LLMs')

print("=== MAIA — TOKENIZER SEARCH ===")

found = []

for filename in ('tokenizer.json', 'tokenizer_config.json', 'vocab.json'):
    for path in base.rglob(filename):
        if 'checkpoint-' in str(path):
            continue

        if path.parent not in found:
            found.append(path.parent)

for i, path in enumerate(found):
    print(f'{i}: {path}')

print(f'\nDirectories found: {len(found)}')

=== MAIA — TOKENIZER SEARCH ===
0: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_Stage2/releases/maia-lite-0.2.0-hf
1: /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/outputs/gpt2-medium-micro-sft
2: /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/outputs/gpt2-medium-stage1
3: /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/outputs/gpt2-medium-stage2-best
4: /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/outputs/gpt2-medium-stage2-final
5: /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/outputs/gpt2-medium-stage3-best
6: /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/outputs/gpt2-medium-stage3-final
7: /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/outputs/gpt2-medium-stage4-lora-mps-safe-best
8: /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/outputs/a100-gpt2-medium-lora-best
9: /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/outputs/a100-gpt2-medium-lora-final
10: /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/outputs/a10

In [ ]:
tokenizer=None
if ENABLE_REAL_TOKEN_COUNT:
    from transformers import AutoTokenizer
    assert TOKENIZER_PATH and Path(TOKENIZER_PATH).is_dir(), 'Set TOKENIZER_PATH to the frozen Maia tokenizer directory.'
    tokenizer=AutoTokenizer.from_pretrained(TOKENIZER_PATH,local_files_only=True,trust_remote_code=False,use_fast=True)
    assert tokenizer.is_fast,'A fast tokenizer is recommended for this full scan'
    print('Tokenizer loaded:',TOKENIZER_PATH,'vocab:',len(tokenizer))
else:
    print('Real tokenizer count disabled; the report will state NOT RUN.')

Tokenizer loaded: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer vocab: 50257


## 4. Full scan — explicitly opt in

Validates each record, CRC/JSON, SHA-256 of `re.sub(r"\s+", " ", text.casefold()).strip()`, source/job/language metadata, proxy count (`max(1, round(len(text)/4))`), UTF-8 bytes, quarantine flags and exact duplicates across the corpus.

A corrupt chunk is **not marked completed**. A failed chunk's transaction is rolled back, including hashes, and logged separately. Fixing source chunks is **not** automatic. An audit may stop on the first corrupted gzip; it is safer than silently marking partial results complete.

**Important:** Running 21,021 chunks can take hours and require several GB of temporary disk for SQLite. Keep acquisition stopped. Source files are never written.

In [ ]:
def norm_text(s):return re.sub(r'\s+',' ',s.casefold()).strip()
def expected_proxy(s):return max(1,round(len(s)/4))
def add_issue(kind,n=1):
    conn.execute('INSERT INTO issue_counts(kind,n) VALUES (?,?) ON CONFLICT(kind) DO UPDATE SET n=n+excluded.n',(kind,n))
def log_example(path,line,kind,detail):
    if conn.execute('SELECT COUNT(*) FROM error_examples').fetchone()[0]<MAX_ERRORS_LOGGED:
        conn.execute('INSERT INTO error_examples(file,line,kind,detail) VALUES (?,?,?,?)',(path.name,line,kind,str(detail)[:250]))

def audit_chunk(jid,path):
    st=path.stat()
    if conn.execute('SELECT 1 FROM completed WHERE name=?',(path.name,)).fetchone():return None
    docs=proxy=real=duplicates=issues=0
    conn.execute('BEGIN IMMEDIATE')
    try:
        with gzip.open(path,'rt',encoding='utf-8') as fh:
            for line_no,line in enumerate(fh,1):
                try:
                    row=json.loads(line)
                    if not isinstance(row,dict):raise ValueError('JSON not an object')
                    text=row.get('text')
                    if not isinstance(text,str):raise ValueError('missing text')
                    digest=hashlib.sha256(norm_text(text).encode('utf-8')).hexdigest()
                    checks={
                        'sha256_mismatch':row.get('normalized_sha256')!=digest,
                        'job_id_mismatch':row.get('job_id')!=jid,
                        'language_mismatch':row.get('expected_language')!=JOB_META[jid]['language'],
                        'text_bytes_mismatch':row.get('text_bytes')!=len(text.encode('utf-8')),
                        'proxy_tokens_mismatch':row.get('proxy_tokens')!=expected_proxy(text),
                        'rights_status_mismatch':row.get('rights_status')!='pending_review',
                        'training_eligibility_mismatch':row.get('eligible_for_training') is not False,
                    }
                    for kind,failed in checks.items():
                        if failed:
                            issues+=1;add_issue(kind);log_example(path,line_no,kind,'field mismatch')
                    inserted=conn.execute('INSERT OR IGNORE INTO seen(digest) VALUES (?)',(digest,)).rowcount
                    if not inserted:duplicates+=1;add_issue('exact_duplicate')
                    docs+=1
                    proxy+=expected_proxy(text)
                    if tokenizer is not None:
                        real+=len(tokenizer.encode(text,add_special_tokens=False))
                except Exception as e:
                    issues+=1;add_issue('invalid_record');log_example(path,line_no,'invalid_record',repr(e))
        # Check source file did not change during read.
        after=path.stat()
        if (st.st_size,st.st_mtime_ns)!=(after.st_size,after.st_mtime_ns):
            raise RuntimeError('Source file changed during audit')
        conn.execute('INSERT INTO completed VALUES (?,?,?,?,?,?,?,?,?,?)',(
            path.name,jid,st.st_size,st.st_mtime_ns,docs,proxy,
            real if tokenizer is not None else None,duplicates,issues,
            datetime.now(timezone.utc).isoformat()))
        conn.commit()
        return docs,proxy,real,duplicates,issues
    except Exception:
        conn.rollback()
        raise

if not ENABLE_FULL_SCAN:
    print('FULL SCAN OFF. Set ENABLE_FULL_SCAN=True in configuration, rerun setup + DB cells, then run this cell.')
else:
    last_snapshot=time.monotonic();since_snapshot=0;scanned=0
    try:
        for jid,path in files:
            if conn.execute('SELECT 1 FROM completed WHERE name=?',(path.name,)).fetchone():continue
            result=audit_chunk(jid,path)
            if result is None:continue
            scanned+=1;since_snapshot+=1
            if scanned%PRINT_EVERY_CHUNKS==0:
                print('New chunks:',scanned,'total completed:',conn.execute('SELECT COUNT(*) FROM completed').fetchone()[0],
                      'last:',path.name,'RSS GB:',round(psutil.Process().memory_info().rss/2**30,2),flush=True)
            if since_snapshot>=CHECKPOINT_EVERY_CHUNKS or time.monotonic()-last_snapshot>=CHECKPOINT_EVERY_MINUTES*60:
                save_checkpoint();since_snapshot=0;last_snapshot=time.monotonic()
    except BaseException as exc:
        print('AUDIT INTERRUPTED:',repr(exc),'Last file:',path.name,flush=True)
        print('Already committed local chunks remain in SQLite; last Drive snapshot may lag.')
        raise
    else:
        save_checkpoint()
        print('Full scan loop finished. Validate totals below.')

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1807 > 1024). Running this sequence through the model will result in indexing errors


New chunks: 25 total completed: 25 last: fineweb2_pt_chunk_000024.jsonl.gz RSS GB: 0.64
New chunks: 50 total completed: 50 last: fineweb2_pt_chunk_000049.jsonl.gz RSS GB: 0.64
New chunks: 75 total completed: 75 last: fineweb2_pt_chunk_000074.jsonl.gz RSS GB: 0.64
New chunks: 100 total completed: 100 last: fineweb2_pt_chunk_000099.jsonl.gz RSS GB: 0.64
New chunks: 125 total completed: 125 last: fineweb2_pt_chunk_000124.jsonl.gz RSS GB: 0.64
New chunks: 150 total completed: 150 last: fineweb2_pt_chunk_000149.jsonl.gz RSS GB: 0.64
New chunks: 175 total completed: 175 last: fineweb2_pt_chunk_000174.jsonl.gz RSS GB: 0.64
New chunks: 200 total completed: 200 last: fineweb2_pt_chunk_000199.jsonl.gz RSS GB: 0.64
Checkpoint saved: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001/quality_audit_full_with_tokenizer_v1/audit_state.sqlite3 chunks: 200
New chunks: 225 total completed: 225 last: fineweb2_pt_chunk_000224.jsonl.gz RSS GB: 0.64
New chunks: 250 to

## 5. Summaries and invariant checks

A successful full scan requires: every inventoried chunk marked completed, zero invalid records / field inconsistencies, zero gzip failures, and no unexpected source mutation. Exact duplicate counts are reported separately; whether duplicates should block training is a project decision.

This stage does **not** infer a missing real-token count from proxy tokens.

In [3]:
done=conn.execute('SELECT COUNT(*) FROM completed').fetchone()[0]
summary=pd.read_sql_query("""SELECT job_id, COUNT(*) AS chunks, SUM(docs) AS documents,
SUM(proxy_tokens) AS proxy_tokens, SUM(real_tokens) AS real_tokens,
SUM(duplicates) AS exact_duplicates, SUM(issues) AS integrity_issues
FROM completed GROUP BY job_id ORDER BY job_id""",conn)
errors=pd.read_sql_query('SELECT kind,n FROM issue_counts ORDER BY n DESC',conn)
print('Completed:',done,'/',len(files))
display(summary)
display(errors)
summary.to_csv(OUT/'full_source_summary.csv',index=False)
errors.to_csv(OUT/'full_issue_counts.csv',index=False)
if done==len(files):
    print('Proxy total:',int(summary.proxy_tokens.sum()))
    print('Documents:',int(summary.documents.sum()))
    print('Difference vs target 8.2B:',8_200_000_000-int(summary.proxy_tokens.sum()))
else:
    print('PARTIAL: totals describe completed chunks only.')
print('Actual tokenizer:', 'ENABLED' if ENABLE_REAL_TOKEN_COUNT else 'NOT RUN')

NameError: name 'conn' is not defined

## 6. Independent language identification (bounded sample)

Runs on evenly distributed source chunks, independent of the full audit state. It does **not** inspect every document. It stores only aggregate results, not document text. Language-ID confidence is imperfect for multilingual, code-heavy, and short documents.

In [ ]:
if ENABLE_LANGUAGE_SAMPLE:
    import langid
    rows=[]
    for jid,paths in files_by_job.items():
        count=min(LANGUAGE_SAMPLE_FILES_PER_JOB,len(paths))
        idx=sorted(set(round(i*(len(paths)-1)/(count-1)) for i in range(count))) if count>1 else [0]
        for k in idx:
            path=paths[k];used=0
            try:
                with gzip.open(path,'rt',encoding='utf-8') as fh:
                    for line in fh:
                        row=json.loads(line)
                        text=row.get('text','')
                        if not isinstance(text,str) or len(text)<150:continue
                        pred,score=langid.classify(text[:3000])
                        rows.append(dict(job_id=jid,expected=JOB_META[jid]['language'],predicted=pred,match=pred==JOB_META[jid]['language']))
                        used+=1
                        if used>=LANGUAGE_SAMPLE_DOCS_PER_FILE:break
            except Exception as e:
                print('Language sample failed:',path.name,repr(e))
    lang_df=pd.DataFrame(rows)
    if len(lang_df):
        lang_summary=lang_df.groupby('job_id').agg(samples=('match','size'),match_rate=('match','mean')).reset_index()
        lang_summary['match_rate_pct']=(lang_summary.match_rate*100).round(2)
        display(lang_summary)
        lang_summary.to_csv(OUT/'language_sample_summary.csv',index=False)
        lang_df.groupby(['job_id','predicted']).size().reset_index(name='count').to_csv(OUT/'language_sample_predictions.csv',index=False)
else:
    print('Language identification NOT RUN.')

## 7. Release gate and machine-readable final report

**Production ready is always `False`** until a separately documented human rights review, benchmark contamination check, near-duplicate strategy, tokenizer calibration and quality acceptance are completed. Do not change source quarantine flags from this notebook.

In [ ]:
done=conn.execute('SELECT COUNT(*) FROM completed').fetchone()[0]
issue_map={r[0]:r[1] for r in conn.execute('SELECT kind,n FROM issue_counts')}
all_complete=done==len(files)
integrity_issues=sum(v for k,v in issue_map.items() if k!='exact_duplicate')
real_tokens=int(summary.real_tokens.sum()) if ENABLE_REAL_TOKEN_COUNT and all_complete else None
report={
 'audit':'maia_corpus_v8_full_validation_v1',
 'utc':datetime.now(timezone.utc).isoformat(),
 'run_id':RUN_ID,'inventory_sha256':inventory_digest,
 'chunks_expected':len(files),'chunks_completed':done,
 'full_integrity_verified':bool(all_complete and integrity_issues==0),
 'total_documents_in_completed_chunks':int(summary.documents.sum()) if len(summary) else 0,
 'total_proxy_tokens_in_completed_chunks':int(summary.proxy_tokens.sum()) if len(summary) else 0,
 'total_real_tokens':real_tokens,
 'real_token_count_status':('COMPLETE' if real_tokens is not None else 'NOT_RUN_OR_INCOMPLETE'),
 'exact_duplicate_count':int(issue_map.get('exact_duplicate',0)),
 'integrity_issues':int(integrity_issues),'issue_counts':issue_map,
 'language_identification':'SAMPLED' if ENABLE_LANGUAGE_SAMPLE and 'lang_df' in globals() and len(lang_df) else 'NOT_RUN',
 'near_duplicate_full_corpus':'NOT_RUN',
 'benchmark_contamination':'NOT_RUN',
 'rights_review':'PENDING',
 'production_ready':False,
 'blockers':['Human license/provenance review','Full near-duplicate review','Benchmark contamination audit',
             'Scientific quality acceptance criteria','Wikipedia PT quota shortfall']+
            ([] if real_tokens is not None else ['Frozen Maia tokenizer full count not completed'])+
            ([] if all_complete and integrity_issues==0 else ['Full corpus integrity not yet verified'])
}
(OUT/'full_validation_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps({k:report[k] for k in ('chunks_expected','chunks_completed','full_integrity_verified','total_documents_in_completed_chunks','total_proxy_tokens_in_completed_chunks','total_real_tokens','exact_duplicate_count','integrity_issues','production_ready')},indent=2))
print('Report:',OUT/'full_validation_report.json')
print('Source chunks remain unchanged.')

## 8. Operational guidance

**First run:** mount Drive, inventory, choose tokenization mode, initialize database, then set `ENABLE_FULL_SCAN=True` and run the full-scan cell. The cell writes periodic Drive snapshots. If the session disconnects, rerun setup/inventory/DB, and rerun the full-scan cell; completed chunks in the snapshot will be skipped.

**If disk is low:** stop, preserve the Drive snapshot, and move to a runtime with more SSD. The `seen` table must not be deleted during an audit; deleting it would invalidate global deduplication.

**If you initially ran without tokenization:** the full scan can finish and establish integrity. For exact full token counts, change to a **new output directory** and rerun with the frozen Maia tokenizer enabled, or develop a separate token-only pass; do not mix incompatible database states.

**If a chunk fails:** save the error and report it. Do not rewrite or delete original source files as part of this audit.

**Interpretation:** 8.2B is a proxy-token target; the actual count may be materially different. Training approval remains blocked pending rights and scientific validation.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [13]:
!python /content/drive/MyDrive/Colab_LLMs/maia_corpus_v8_freeze.py


FREEZE CREATED: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001/freeze_v8_001
Manifest: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001/freeze_v8_001/manifest.json
Report: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001/freeze_v8_001/audit_report.md
Checksums: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001/freeze_v8_001/SHA256SUMS.txt
STATUS: AUDIT_PASSED_TOKENIZER_PROVENANCE_PENDING
Tokenizer fingerprint: 5da873b1198ebe273f4254be9a67a5e12426c796c6026022e8a65c57113a2f07
No original audit data were modified.
